In [11]:
import json
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import pickle

In [13]:
pd.set_option('display.max_columns',None)

In [15]:
# Load PKL

def load_pkl(path):
    with open(path, "rb") as f:
        return pickle.load(f)

In [33]:
eval_data = load_pkl("shortest_path_external_graphs.pkl")

print(type(eval_data))
print("Number of items:", len(eval_data))

print("\nFirst item type:")
print(type(eval_data[0]))

print("\nFirst item:")
print(eval_data[0])

<class 'list'>
Number of items: 30

First item type:
<class 'dict'>

First item:
{'G': <networkx.classes.digraph.DiGraph object at 0x000002D2778D9D30>, 's': 0, 't': 5, 'density': 3.0834914611005693, 'n': 1054, 'm': 3250, 'source_original': 1, 'sink_original': 6, 'network_name': 'rail_road_transportation_network'}


In [35]:
print(df_external.shape)
print(df_external[[
    "source",
    "sink",
    "shortest_path_length",
    "shortest_path_edges"
]])

(30, 16)
    source  sink  shortest_path_length  shortest_path_edges
0        0     5            694.054024                   63
1        0     6            184.159473                   22
2        0     7            159.905843                   24
3        0     8            160.656519                   26
4        0   974            803.250811                   62
5        1     5            418.984733                   35
6        1     6            201.887737                   22
7        1     7            274.522361                   39
8        1     8            274.912421                   39
9        1   974            574.482408                   47
10       2     5            189.226226                   20
11       2     6            368.826401                   27
12       2     7            477.858833                   52
13       2     8            478.248894                   52
14       2   974            344.723901                   32
15       3     5            534

In [19]:
def evaluation_graph_characteristics(sample, dataset_name):

    G = sample["G"]
    source = sample["s"]
    sink = sample["t"]

    n = sample["n"]
    m = sample["m"]


    # Degree characteristics

    in_degrees = np.array([d for _, d in G.in_degree()])
    out_degrees = np.array([d for _, d in G.out_degree()])
    total_degrees = np.array([d for _, d in G.degree()])


    
    # Find edge weight attribute

    # Check this first -- likely "cost" or "dist"
    first_edge = next(iter(G.edges(data=True)))
    edge_attributes = first_edge[2]

    if "cost" in edge_attributes:
        weight = "cost"
    elif "dist" in edge_attributes:
        weight = "dist"
    else:
        weight = None


    
    # Original shortest path

    try:

        sp_length = nx.shortest_path_length(G, source=source, target=sink, weight=weight)

        sp = nx.shortest_path(G, source=source, target=sink, weight=weight)

        sp_edges = len(sp) - 1

    except nx.NetworkXNoPath:

        sp_length = np.nan
        sp_edges = np.nan


    
    # Edge values

    if weight is not None:
        costs = [data[weight] for _, _, data in G.edges(data=True)]

        mean_cost = np.mean(costs)
        std_cost = np.std(costs)

    else:
        mean_cost = np.nan
        std_cost = np.nan


    
    # Return one graph-level row
    
    return {
        "dataset": dataset_name,

        "graph_seed": sample.get("seed", np.nan),
        "replication": sample.get("rep", np.nan),

        "n_nodes": n,
        "n_arcs": m,
        "density": sample.get("density", m / n),

        "source": source,
        "sink": sink,

        "avg_in_degree": in_degrees.mean(),
        "avg_out_degree": out_degrees.mean(),
        "std_degree": total_degrees.std(),
        "max_degree": total_degrees.max(),

        "shortest_path_length": sp_length,
        "shortest_path_edges": sp_edges,

        "mean_cost": mean_cost,
        "std_cost": std_cost}

In [21]:
# Convert entire dataset

def summarize_evaluation_dataset(data, dataset_name):

    rows = [evaluation_graph_characteristics(sample, dataset_name) for sample in data]

    return pd.DataFrame(rows)

In [37]:
# Load evaluation datasets
id_eval = load_pkl("shortest_path_id_new_graphs.pkl")
size_ood_eval = load_pkl("shortest_path_ood_size_graphs.pkl")
value_ood_eval = load_pkl("shortest_path_value_ood_graphs.pkl")
wood_eval = load_pkl("shortest_path_wood_graphs.pkl")
external_eval = load_pkl("shortest_path_external_graphs.pkl")


# Convert evaluation datasets to DataFrames
df_id = summarize_evaluation_dataset(id_eval, "ID")
df_size_ood = summarize_evaluation_dataset(size_ood_eval,"Size OOD")
df_value_ood = summarize_evaluation_dataset(value_ood_eval, "Value OOD")
df_wood = summarize_evaluation_dataset(wood_eval, "Wood")
df_external = summarize_evaluation_dataset(external_eval, "External")

In [39]:
df_id_unique = df_id.drop_duplicates(subset=["graph_seed"])
df_size_ood_unique = df_size_ood.drop_duplicates(subset=["graph_seed"])
df_value_ood_unique = df_value_ood.drop_duplicates(subset=["graph_seed"])
df_wood_unique = df_wood.drop_duplicates(subset=["graph_seed"])

In [53]:
all_data = pd.concat([df_id_unique, df_size_ood_unique, df_value_ood_unique, df_wood_unique, df_external], ignore_index=True)


metrics = ["shortest_path_length", "shortest_path_edges", "n_nodes", "n_arcs"]


summary = (all_data.groupby("dataset")[metrics].agg(["mean", "median"]))

summary.round(3)

shortest_path_length          shortest_path_edges          n_nodes  \
                          mean   median                mean median      mean   
dataset                                                                        
External               364.593  354.132              36.333   33.5  1054.000   
ID                      15.314   13.000               3.900    4.0    51.667   
Size OOD                20.600   17.000               5.506    5.0   325.000   
Value OOD               92.972   79.500               3.608    3.0    51.667   
Wood                    15.375   14.000              10.350   10.0    86.500   

                     n_arcs          
           median      mean  median  
dataset                              
External   1054.0  3250.000  3250.0  
ID           50.0   193.750   165.0  
Size OOD    300.0  1416.071  1100.0  
Value OOD    50.0   193.750   165.0  
Wood         74.5   347.500   292.0

In [ ]:
# 1) redo onein (100%) and mixed topo (60/10/10/10/10) - all synthetic
# 2) create onein filtered (100%) - synthetic 
# 3) redo onein filtered with mixed topo (60/10/10/10/10) -synthetic
# 4) redo one in filtered with mixed topo AND with real world (50/10/10/10/10/10) - synthetic + real world